# Dynamic Models

Use `@wrap_model_call` middleware to swap models mid-conversation based on state (here: message count).

**Key learnings**
- Short threads can use a cheap/fast model
- Longer threads can switch to a larger-context model
- `request.override(model=...)` changes the model for that call only


In [ ]:
from dotenv import load_dotenv

load_dotenv()

### State-based model selection

Count messages in `request.messages`; if the conversation is long, route to Claude, otherwise stay on the efficient default.


In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model("claude-sonnet-4-6")
standard_model = init_chat_model("gpt-5-nano")


@wrap_model_call
def state_based_model(request: ModelRequest,
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    message_count = len(request.messages)

    if message_count > 10:
        # Long conversation — prefer a larger context window
        model = large_model
    else:
        # Short conversation — keep costs/latency low
        model = standard_model

    request = request.override(model=model)
    return handler(request)


### Create the agent

The default model is still required; middleware may override it per call.


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

### Short conversation → standard model

A single user message stays under the threshold — check `response_metadata["model_name"]`.


In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

In [ ]:
print(response["messages"][-1].response_metadata["model_name"])

### Long conversation → large model

Feed a long history (>10 messages) and confirm the larger model is selected.


In [ ]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

In [ ]:
print(response["messages"][-1].response_metadata["model_name"])